# 2 - Model Comparison

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

from src.utils import (
    load_historical,
    make_pipeline,
    infer_feature_types,
    RANDOM_STATE,
    compare_topx_levels
)

In [2]:
X, y, df = load_historical("../data/historical_data.csv")

print("Shape:", df.shape)
df.head()

Shape: (12000, 18)


,id,day,event_type,category,region,device,account_age_days,num_prev_listings,prev_reports_30d,verification_level,price,num_images,message_length,contains_off_platform,urgency_words,payment_attempt,time_to_first_response_min,is_suspicious
0,0,8,ad_post,other,urban,android,38.4,2,1,1,594.16,3,91,0,1,0,2.3,0
1,1,4,ad_post,fashion,urban,android,20.0,1,0,1,134.47,2,150,0,0,0,13.6,0
2,2,4,ad_post,other,metro,ios,46.7,3,1,1,198.52,3,72,0,0,0,4.2,0
3,3,3,ad_post,furniture,metro,android,44.3,3,1,2,141.20,3,0,0,0,0,19.8,0
4,4,1,ad_post,electronics,metro,web,211.2,5,0,0,81.39,3,9,0,0,1,23.3,0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train suspicious ratio:", y_train.mean())
print("Test suspicious ratio:", y_test.mean())

Train shape: (9600, 16)
Test shape: (2400, 16)
Train suspicious ratio: 0.10197916666666666
Test suspicious ratio: 0.10208333333333333


In [4]:
baseline_pipe = make_pipeline(
    DummyClassifier(strategy="most_frequent"),
    X_train
)

print(baseline_pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['day', 'account_age_days',
                                                   'num_prev_listings',
                                                   'prev_reports_30d',
                                                   'verification_level',
                                                   'price', 'num_images',
                                                   'message_length',
                                                   'contains_off_platform',
                                                   'urgency_

In [5]:
models = {
    "baseline": DummyClassifier(strategy="most_frequent"),
    "logreg": LogisticRegression(max_iter=2000, solver="liblinear"),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    pipe = make_pipeline(model, X_train)

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "ap": "average_precision",
            "roc_auc": "roc_auc",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1"
        },
        n_jobs=-1
    )

    results.append({
        "model": name,
        "cv_ap_mean": scores["test_ap"].mean(),
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
        "cv_precision_mean": scores["test_precision"].mean(),
        "cv_recall_mean": scores["test_recall"].mean(),
        "cv_f1_mean": scores["test_f1"].mean()
    })

results_df = pd.DataFrame(results).sort_values("cv_ap_mean", ascending=False)

best_model_name = results_df.iloc[0]["model"]
print("Best model from CV:", best_model_name)

Best model from CV: logreg


In [7]:
results_df

,model,cv_ap_mean,cv_roc_auc_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean
1,logreg,0.286193,0.738968,0.584314,0.042909,0.079580
2,random_forest,0.249786,0.731484,0.497619,0.013276,0.025765
0,baseline,0.101979,0.500000,0.000000,0.000000,0.000000


In [8]:
best_model_name = results_df.iloc[0]["model"]
print("Best model from CV:", best_model_name)

best_model = models[best_model_name]

best_pipe = make_pipeline(best_model, X_train)
best_pipe.fit(X_train, y_train)

y_proba_test = best_pipe.predict_proba(X_test)[:, 1]

print("Test AP:", average_precision_score(y_test, y_proba_test))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba_test))

Best model from CV: logreg
Test AP: 0.2759987485768812
Test ROC-AUC: 0.7399933708982434


In [9]:
from src.utils import compare_topx_levels

topx_results = compare_topx_levels(y_test, y_proba_test)

print(topx_results)

   top_frac  k_flagged  precision    recall
0      0.03         72   0.430556  0.126531
1      0.05        120   0.366667  0.179592
2      0.10        240   0.304167  0.297959


## Metric motivation:

Vi valde att prioritera Average Precision eftersom datasetet är obalanserat och arbetsflödet i verksamheten fokuserar på att granska fallen med högst risk. Detta mått utvärderar hur bra modellen rankar misstänkta transaktioner samtidigt som den behåller hög precision bland de fall som flaggas.